In [ ]:
!pip install osmnx networkx
df = spark.read.csv("s3a://raw/porto-trips/train.csv", header=True)
df.distinct("

In [2]:
import osmnx as ox

# Télécharger le graphe routier réel
G = ox.graph_from_place("Casablanca, Morocco", network_type="drive")

print("Graph chargé")

Graph chargé


In [ ]:
def porto_to_casa_point(lon, lat):
    lon_c, lat_c = porto_to_casa_linear(lon, lat)
    return lat_c, lon_c  # attention ordre lat, lon

In [ ]:
import networkx as nx

def compute_real_route(porto_polyline):
    pts = json.loads(porto_polyline)

    # début / fin du trajet Porto
    start_lon, start_lat = pts[0]
    end_lon, end_lat = pts[-1]

    # mapping vers Casablanca
    start_lat_c, start_lon_c = porto_to_casa_point(start_lon, start_lat)
    end_lat_c, end_lon_c = porto_to_casa_point(end_lon, end_lat)

    # trouver les noeuds les plus proches
    start_node = ox.distance.nearest_nodes(G, start_lon_c, start_lat_c)
    end_node = ox.distance.nearest_nodes(G, end_lon_c, end_lat_c)

    # calcul du chemin réel
    route = nx.shortest_path(G, start_node, end_node, weight="length")

    # convertir en coordonnées GPS
    route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in route]

    return route_coords

In [ ]:
def plot_real_route(route_coords):
    import folium

    m = folium.Map(location=route_coords[0], zoom_start=13)

    folium.PolyLine(route_coords, color="green", weight=5).add_to(m)

    return m

In [ ]:
row = df.select("POLYLINE").limit(1).collect()[0]

route = compute_real_route(row["POLYLINE"])

m = plot_real_route(route)
m